# 02 — Nettoyage

**Source :** [Online Retail sur Kaggle](https://www.kaggle.com/datasets/carrie1/ecommerce-data), données d'origine UCI, licence CC BY 4.0


**Questions métier :**
1. Quels pays, hors Royaume-Uni, rapportent le plus de chiffre d'affaires, et quel continent domine ?
2. Comment le chiffre d'affaires évolue-t-il mois par mois ? Y a-t-il un pic avant Noël ?
3. Quels jours de la semaine et quelles heures concentrent le plus de commandes ?
4. Quels produits rapportent le plus de chiffre d'affaires ?

**Objectif de ce notebook :** Nettoyer les données pour analyse


In [1]:
import pandas as pd

In [2]:
vente = pd.read_csv("../data/raw/online_retail.csv", parse_dates=["InvoiceDate"], dtype={"InvoiceNo": "str", "StockCode": "str", "CustomerID": "str"}, encoding="latin-1")
vente.head(15)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom


**Supression des doublons** : Ce sont les mêmes facture, même produits, même heure, même prix

In [3]:
print(len(vente))
vente = vente.drop_duplicates()
print(len(vente))

541909
536641


**Description vide** : Supprimer les lignes inconnus car il y a que 0.3% de lignes inconnus

In [4]:
print(len(vente))
vente = vente.dropna(subset=["Description"])
print(len(vente))

536641
535187


**Quantité** : Conserver les quantité > 0 car les quantité < ne sont pas des ventes

In [5]:
print(len(vente))
vente = vente[vente["Quantity"] > 0]
print(len(vente))

535187
525462


**Prix** : Conserver les prix > 0 car les autres prix faussent les calculs

In [6]:
print(len(vente))
vente = vente[vente["UnitPrice"] > 0]
print(len(vente))

525462
524878


**StockCode** :  Tout mettre en majuscule pour que les codes soient les plus uniforme possible

In [7]:
print(vente["StockCode"].nunique())
vente["StockCode"] = vente["StockCode"].str.upper()
print(vente["StockCode"].nunique())

3922
3812


**CustomerID** : Garder les lignes et remplir celles manquantes par inconnu. Cette colonne n'est pas indispensable aux réponses de nos questions 

In [8]:
print(vente["CustomerID"].isna().sum())
vente["CustomerID"] = vente["CustomerID"].fillna("Inconnu")
print(vente["CustomerID"].isna().sum())

132186
0


**Chiffres d'affaire** : Creer une colone chiffre d'affaire CA = Quantity × UnitPrice. Elle sera essentielle pour répondre à nos questions

In [9]:
vente["CA"] = (vente["Quantity"] * vente["UnitPrice"]).round(2)

**Valeurs aberrantes** : Garde ce valeurs car il existe des commandes de grossiste rares mais réelles

In [10]:
vente["CA"].describe()
q1 = vente["CA"].quantile(0.25)
q3 = vente["CA"].quantile(0.75)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart
tres_chers = vente[vente["CA"] > borne_haute]
len(tres_chers)

42624

**Les plus grosses commandes**

In [11]:
vente.sort_values("Quantity", ascending=False).head(5)   

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,CA
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,168469.60
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom,77183.60
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901,United Kingdom,1008.00
206121,554868,22197,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,0.72,13135,United Kingdom,3096.00
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,0.82,18087,United Kingdom,3202.92


**Commandes géantes** : Supprimer les factures 581483 et 541431, annulées quelques minutes après (C581484, C541433)

In [12]:
print(len(vente))
vente = vente[~vente["InvoiceNo"].isin(["581483", "541431"])]
print(len(vente))

524878
524876


**Enregistrement** : Sauvegarder le fichier propre dans data/processed

In [13]:
vente.to_csv("../data/processed/online_retail_propre.csv", index=False)   

## Journal de nettoyage

| Quoi | Pourquoi | Lignes |
|---|---|---|
| Doublons supprimés | Double saisie | 5 268 |
| Description vide supprimée | Produit inconnu (0,3 %) | 1 454 |
| Quantités ≤ 0 supprimées | Pas des ventes | 9 725 |
| Prix ≤ 0 supprimés | Une vente a un prix positif | 584 |
| StockCode en majuscules | Codes uniformes | 110 codes fusionnés |
| CustomerID vide → « Inconnu » | Pas utile aux questions | 132 186 remplis |
| CA créé et arrondi | Quantity × UnitPrice | — |
| CA au-dessus de la borne gardé | Grossistes, rares mais réels | 42 624 gardées |
| Commandes géantes supprimées | Annulées juste après | 2 |
| **Total** | 541 909 → 524 876 | **17 033 supprimées** |